# L2: Edge Vector Memory with Qdrant

In this lesson, you'll set up **Qdrant Edge** for local vector storage on a device. Qdrant Edge is a lightweight, embedded vector search engine that runs in-process, with no server required. Data stays on-device, giving you low latency and full offline capability.

You'll learn to:
- Install and configure Qdrant Edge
- Create an EdgeShard for local vector storage
- Insert vectors with metadata payloads
- Run nearest-neighbor queries on-device
- Persist and reload data from disk

## Setup

In [ ]:
!pip install qdrant-edge-py

## 1. Create a Storage Directory

Qdrant Edge stores vectors in a local directory called an **EdgeShard**. Think of it as a self-contained vector database that lives on your device's filesystem.

In [ ]:
from pathlib import Path

SHARD_DIRECTORY = "./qdrant-edge-shard"
Path(SHARD_DIRECTORY).mkdir(parents=True, exist_ok=True)
print(f"Shard directory ready: {SHARD_DIRECTORY}")

## 2. Configure the EdgeShard

Before creating a shard, you define what kind of vectors it will store. This includes:
- **Vector name**: a label for the vector field
- **Dimension**: the size of each vector (must match your embedding model)
- **Distance metric**: how similarity is measured (Cosine, Euclidean, or Dot)

In [ ]:
from qdrant_edge import Distance, EdgeConfig, VectorDataConfig

VECTOR_NAME = "memory"
VECTOR_DIMENSION = 384  # Common dimension for small embedding models

config = EdgeConfig(
    vector_data={
        VECTOR_NAME: VectorDataConfig(
            size=VECTOR_DIMENSION,
            distance=Distance.Cosine,
        )
    }
)

print(f"Configured EdgeShard: {VECTOR_NAME} ({VECTOR_DIMENSION}d, Cosine)")

## 3. Initialize the EdgeShard

Now create the shard. This is your on-device vector database.

In [ ]:
from qdrant_edge import EdgeShard

edge_shard = EdgeShard(SHARD_DIRECTORY, config)
print("EdgeShard initialized")
print(f"Info: {edge_shard.info()}")

## 4. Insert Vectors

Each vector is stored as a **Point** with:
- An **id** (integer)
- A **vector** dictionary mapping vector names to float arrays
- An optional **payload** dictionary for metadata (timestamps, labels, locations, etc.)

For this demo, we'll use random vectors. In a real application, these come from an embedding model.

In [ ]:
import numpy as np
import time
from qdrant_edge import Point, UpdateOperation

# Simulate 100 memory entries from a device
np.random.seed(42)
num_points = 100

points = []
categories = ["object", "face", "scene", "text", "audio"]

for i in range(num_points):
    vector = np.random.randn(VECTOR_DIMENSION).astype(np.float32).tolist()
    point = Point(
        id=i,
        vector={VECTOR_NAME: vector},
        payload={
            "category": categories[i % len(categories)],
            "timestamp": time.time() - (num_points - i) * 60,  # 1 min apart
            "device": "glasses",
            "confidence": round(np.random.uniform(0.5, 1.0), 2),
        }
    )
    points.append(point)

# Upsert all points in one batch
edge_shard.update(UpdateOperation.upsert_points(points))
print(f"Inserted {num_points} points into EdgeShard")

## 5. Query Vectors

Search for the nearest neighbors to a query vector. This runs entirely on-device with no network call.

In [ ]:
from qdrant_edge import Query, QueryRequest

# Create a query vector (in practice, this comes from your embedding model)
query_vector = np.random.randn(VECTOR_DIMENSION).astype(np.float32).tolist()

results = edge_shard.query(
    QueryRequest(
        query=Query.Nearest(query_vector, using=VECTOR_NAME),
        limit=5,
        with_vector=False,
        with_payload=True,
    )
)

print("Top 5 nearest neighbors:")
for result in results:
    print(f"  id={result.id}, score={result.score:.4f}, payload={result.payload}")

## 6. Retrieve Points by ID

You can also fetch specific points by their ID, useful when you already know which memory entry you need.

In [ ]:
retrieved = edge_shard.retrieve(
    point_ids=[0, 1, 2],
    with_payload=True,
    with_vector=False,
)

for point in retrieved:
    print(f"Point {point.id}: {point.payload}")

## 7. Measure Query Latency

On-device queries should be fast. Let's benchmark query performance at different scales to understand how Qdrant Edge handles growth.

In [ ]:
import time

# Benchmark at current scale (100 points)
def benchmark_queries(shard, dim, n_runs=100):
    latencies = []
    for _ in range(n_runs):
        q = np.random.randn(dim).astype(np.float32).tolist()
        start = time.perf_counter()
        shard.query(
            QueryRequest(
                query=Query.Nearest(q, using=VECTOR_NAME),
                limit=5,
                with_vector=False,
                with_payload=True,
            )
        )
        latencies.append((time.perf_counter() - start) * 1000)
    return np.array(latencies)

latencies = benchmark_queries(edge_shard, VECTOR_DIMENSION)
print(f"Query latency at {num_points} points ({VECTOR_DIMENSION}d):")
print(f"  Mean:  {latencies.mean():.2f} ms")
print(f"  P50:   {np.percentile(latencies, 50):.2f} ms")
print(f"  P95:   {np.percentile(latencies, 95):.2f} ms")
print(f"  P99:   {np.percentile(latencies, 99):.2f} ms")

## 8. Scale Test: 10,000 Vectors

A real device accumulates thousands of memories over days and weeks. Let's scale up to 10,000 vectors and verify that query latency stays under 20ms (the Context Hub target).

In [ ]:
# Scale up: add 10,000 vectors to the same shard
SCALE_COUNT = 10_000

print(f"Inserting {SCALE_COUNT} vectors...")
start = time.perf_counter()

batch_size = 1000
for batch_start in range(num_points, num_points + SCALE_COUNT, batch_size):
    batch_points = []
    for i in range(batch_start, min(batch_start + batch_size, num_points + SCALE_COUNT)):
        vector = np.random.randn(VECTOR_DIMENSION).astype(np.float32).tolist()
        batch_points.append(Point(
            id=i,
            vector={VECTOR_NAME: vector},
            payload={
                "category": categories[i % len(categories)],
                "timestamp": time.time() - (num_points + SCALE_COUNT - i) * 60,
                "device": ["glasses", "phone", "watch"][i % 3],
                "confidence": round(np.random.uniform(0.5, 1.0), 2),
            }
        ))
    edge_shard.update(UpdateOperation.upsert_points(batch_points))

insert_time = time.perf_counter() - start
total = num_points + SCALE_COUNT
print(f"Inserted {SCALE_COUNT} vectors in {insert_time:.1f}s ({SCALE_COUNT/insert_time:.0f} vectors/sec)")
print(f"Total vectors in shard: {total}")

# Benchmark at 10K scale
latencies_10k = benchmark_queries(edge_shard, VECTOR_DIMENSION)
print(f"\nQuery latency at {total} points ({VECTOR_DIMENSION}d):")
print(f"  Mean:  {latencies_10k.mean():.2f} ms")
print(f"  P50:   {np.percentile(latencies_10k, 50):.2f} ms")
print(f"  P95:   {np.percentile(latencies_10k, 95):.2f} ms")
print(f"  P99:   {np.percentile(latencies_10k, 99):.2f} ms")
print(f"\nTarget: < 20ms P95  {'PASS' if np.percentile(latencies_10k, 95) < 20 else 'NEEDS OPTIMIZATION'}")

## 9. Persistence: Close and Reload

Qdrant Edge persists data to disk. You can close the shard and reopen it later without losing data. When reopening, pass `None` for the config since the shard already knows its configuration.

In [ ]:
# Close the shard
edge_shard.close()
print("Shard closed")

# Reopen from disk (no config needed)
edge_shard = EdgeShard(SHARD_DIRECTORY)
print("Shard reopened from disk")

# Verify data is still there
results = edge_shard.query(
    QueryRequest(
        query=Query.Nearest(query_vector, using=VECTOR_NAME),
        limit=3,
        with_vector=False,
        with_payload=True,
    )
)
print(f"Query after reload returned {len(results)} results")
for r in results:
    print(f"  id={r.id}, score={r.score:.4f}")

## 10. Cleanup

In [ ]:
edge_shard.close()

import shutil
shutil.rmtree(SHARD_DIRECTORY, ignore_errors=True)
print("Cleaned up")

## Summary

In this lesson you learned how to:
- Install `qdrant-edge-py` and create an `EdgeShard` for on-device storage
- Configure vector dimensions and distance metrics with `EdgeConfig`
- Insert points with `UpdateOperation.upsert_points()`
- Query nearest neighbors with `Query.Nearest()`
- Scale to 10,000+ vectors while staying under the 20ms P95 latency target
- Persist data to disk and reload it

In the next lesson, you'll generate real embeddings on-device using AI Hub and store them in your EdgeShard.